# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hafsaShaban/flyrank_internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os
if not os.path.exists('/content/flyrank_internship'):
    !git clone https://github.com/hafsaShaban/flyrank_internship.git
%cd /content/flyrank_internship

import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupKFold, KFold

df = pd.read_csv('data/raw/content_refresh_anonymized.csv')
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)
print("Loaded:", len(df), "rows")

Cloning into 'flyrank_internship'...
remote: Enumerating objects: 170, done.
remote: Counting objects: 100% (170/170), done.
remote: Compressing objects: 100% (126/126), done.
remote: Total 170 (delta 70), reused 94 (delta 27), pack-reused 0 (from 0)
Receiving objects: 100% (170/170), 2.28 MiB | 10.08 MiB/s, done.
Resolving deltas: 100% (70/70), done.
/content/flyrank_internship
Loaded: 30000 rows


## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

Finding 1: "The random forest got about 37 of its top 50 right... strong evidence that a learned ranking can beat a fixed rule." The label here (is_declining_label = trend_direction == "down") comes from a rule computed on the current window, not an observed future outcome — the guide itself flags this as a "beginner proxy label," not the ideal capstone target. My methodology question: does client-holdout validation carry this claim as far as it's stated? Client-holdout proves the model generalizes to unseen clients, but it doesn't prove the model predicts genuine future decline — since the label was never a future outcome to begin with, the model may just be learning to reproduce the current-window rule better than the baseline rule does, not learning something predictive.

Finding 2: "The starter's high-confidence label demands enough evidence: final score above the 80th percentile, impressions_90d >= 500, sessions_90d >= 10, and model probability >= 0.50." My methodology question: this compound filter mixes a data-derived threshold (80th percentile, which shifts if the underlying score distribution shifts) with fixed absolute thresholds (500, 10, 0.50) — the guide doesn't state how these fixed cutoffs were chosen or validated against precision at that confidence tier specifically, only Precision@50 for the overall ranking. It's unclear whether "high-confidence" rows are actually more precise than the top-50 as a whole, or just a stricter-sounding filter that hasn't been separately measured.

In [2]:
print("Declining label rate (current-window proxy):", df['is_declining_label'].mean())
print("Rows meeting starter's compound high-confidence-style filter (adapted to our starter columns):")
mask = (df['impressions_90d'] >= 500) & (df['sessions_90d'] >= 10)
print(mask.sum(), "of", len(df))


Declining label rate (current-window proxy): 0.5420666666666667
Rows meeting starter's compound high-confidence-style filter (adapted to our starter columns):
11696 of 30000


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

Re-running the ML-08 random forest under both a random split and a grouped (client-holdout) split, to show the before/after gap — the gap itself is a finding about how much memorization was happening, per the hunting-leakage skill.

In [3]:
numeric_features = ['word_count','char_count','ctr','avg_position','engagement_rate','scroll_rate',
    'ai_traffic_pct','content_age_days','days_since_last_update','search_volume','cpc',
    'impressions_90d','clicks_90d','pageviews_90d','sessions_90d','users_90d',
    'engaged_sessions_90d','ai_sessions_90d','scroll_events_90d',
    'days_with_impressions','days_with_sessions']
categorical_features = ['content_type','main_intent','competition_level','freshness_tier',
    'word_count_tier','char_count_tier','impression_tier','position_tier']

X = df[numeric_features].copy()
for col in numeric_features:
    X[f'has_{col}'] = X[col].notnull().astype(int)
    X[col] = X[col].fillna(0)
X_cat = pd.get_dummies(df[categorical_features], dummy_na=True)
X = pd.concat([X, X_cat], axis=1)
y = df['is_declining_label']
groups = df['client_id']

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

def cv_precision(splitter, X, y, groups=None):
    scores_out = np.zeros(len(y))
    splits = splitter.split(X, y, groups) if groups is not None else splitter.split(X, y)
    for train_idx, test_idx in splits:
        m = RandomForestClassifier(n_estimators=200, random_state=42)
        m.fit(X.iloc[train_idx], y.iloc[train_idx])
        scores_out[test_idx] = m.predict_proba(X.iloc[test_idx])[:, 1]
    return precision_at_k(scores_out, y, 50)

random_p50 = cv_precision(KFold(n_splits=5, shuffle=True, random_state=42), X, y)
grouped_p50 = cv_precision(GroupKFold(n_splits=5), X, y, groups)

print("Precision@50 — random split (before, optimistic):", random_p50)
print("Precision@50 — grouped client-holdout split (after, honest):", grouped_p50)
print("Gap:", random_p50 - grouped_p50)


Precision@50 — random split (before, optimistic): 0.98
Precision@50 — grouped client-holdout split (after, honest): 0.58
Gap: 0.4


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

Re-running the leakage hunt from ML-05 on the final feature set used above — confirming no label-derived, product-flag, or overlapping-window columns snuck in.

In [4]:
leaky_terms = ['trend', 'is_declining', 'health_score', 'priority_score', 'action_type']
leaky_found = [c for c in X.columns if any(term in c.lower() for term in leaky_terms)]
print("Leaky columns found in final feature matrix:", leaky_found)

overlap_terms = ['last_30d']
overlap_found = [c for c in X.columns if any(term in c.lower() for term in overlap_terms)]
print("Overlapping-window columns found:", overlap_found)


Leaky columns found in final feature matrix: []
Overlapping-window columns found: []


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

On this anonymized 30,000-row starter slice, using a client-holdout split, a random-forest-based ranking showed a directionally higher Precision@50 than the transparent baseline rule (0.740 vs 0.240 in the reference pipeline). This is decision-support evidence that a learned ranking may prioritize review candidates more effectively than the fixed rule on this dataset — not a guarantee of real-world refresh outcomes, since the underlying label describes current-window status rather than an observed future decline.

In [5]:
print(f"My honest, grouped-split Precision@50: {grouped_p50:.3f}")
print(f"Base rate: {y.mean():.3f}")
print("This is the number the rewritten claim above should actually cite — not the reference pipeline's number.")


My honest, grouped-split Precision@50: 0.580
Base rate: 0.542
This is the number the rewritten claim above should actually cite — not the reference pipeline's number.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.